In [18]:
from pathlib import Path
import shutil
import random
from sklearn.model_selection import train_test_split

# --------------------------------------------------
# Paths
# --------------------------------------------------

SOURCE_DIR = Path("../data/processed/tomato_potato")
SPLIT_DIR = Path("../data/processed/tomato_potato_split")

print("Source dataset:")
print(SOURCE_DIR.resolve())

print("\nSplit dataset:")
print(SPLIT_DIR.resolve())

print("\nSource exists:", SOURCE_DIR.exists())

Source dataset:
C:\Users\ATHARVA\Desktop\Mini project\ai-farmer-assistant\data\processed\tomato_potato

Split dataset:
C:\Users\ATHARVA\Desktop\Mini project\ai-farmer-assistant\data\processed\tomato_potato_split

Source exists: True


In [19]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Collect all image paths and class labels
# --------------------------------------------------

image_extensions = {".jpg", ".jpeg", ".png"}

records = []

for class_dir in sorted(SOURCE_DIR.iterdir()):

    if not class_dir.is_dir():
        continue

    class_name = class_dir.name

    for image_path in sorted(class_dir.iterdir()):

        if image_path.is_file() and image_path.suffix.lower() in image_extensions:
            records.append({
                "path": str(image_path),
                "class": class_name
            })

df = pd.DataFrame(records)

print("Total images collected:", len(df))
print("Number of classes:", df["class"].nunique())

print("\nImages per class:")
print(df["class"].value_counts().sort_index())

Total images collected: 20298
Number of classes: 13

Images per class:
class
Potato___Early_blight                            1000
Potato___Late_blight                             1000
Potato___healthy                                  152
Tomato___Bacterial_spot                          2127
Tomato___Early_blight                            1000
Tomato___Late_blight                             1901
Tomato___Leaf_Mold                                952
Tomato___Septoria_leaf_spot                      1771
Tomato___Spider_mites Two-spotted_spider_mite    1676
Tomato___Target_Spot                             1404
Tomato___Tomato_Yellow_Leaf_Curl_Virus           5357
Tomato___Tomato_mosaic_virus                      373
Tomato___healthy                                 1585
Name: count, dtype: int64


In [20]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["class"],
    random_state=RANDOM_STATE
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["class"],
    random_state=RANDOM_STATE
)

print("=" * 60)
print("STRATIFIED SPLIT RESULTS")
print("=" * 60)

print("\nTrain images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("\nTotal:", len(train_df) + len(val_df) + len(test_df))

print("\nExpected:")
print("Train ≈ 70%")
print("Validation ≈ 15%")
print("Test ≈ 15%")

STRATIFIED SPLIT RESULTS

Train images: 14208
Validation images: 3045
Test images: 3045

Total: 20298

Expected:
Train ≈ 70%
Validation ≈ 15%
Test ≈ 15%


In [21]:
# --------------------------------------------------
# Verify stratification
# --------------------------------------------------

print("=" * 70)
print("CLASS DISTRIBUTION ACROSS TRAIN / VALIDATION / TEST")
print("=" * 70)

distribution = pd.DataFrame({
    "Total": df["class"].value_counts(),
    "Train": train_df["class"].value_counts(),
    "Validation": val_df["class"].value_counts(),
    "Test": test_df["class"].value_counts()
}).fillna(0).astype(int)

# Calculate percentages
distribution["Train %"] = (
    distribution["Train"] / distribution["Total"] * 100
).round(2)

distribution["Validation %"] = (
    distribution["Validation"] / distribution["Total"] * 100
).round(2)

distribution["Test %"] = (
    distribution["Test"] / distribution["Total"] * 100
).round(2)

print(distribution.sort_index().to_string())

print("\n" + "=" * 70)
print("SPLIT TOTALS")
print("=" * 70)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

print("\nOverall percentages:")
print("Train:", round(len(train_df) / len(df) * 100, 2), "%")
print("Validation:", round(len(val_df) / len(df) * 100, 2), "%")
print("Test:", round(len(test_df) / len(df) * 100, 2), "%")

CLASS DISTRIBUTION ACROSS TRAIN / VALIDATION / TEST
                                               Total  Train  Validation  Test  Train %  Validation %  Test %
class                                                                                                       
Potato___Early_blight                           1000    700         150   150    70.00         15.00   15.00
Potato___Late_blight                            1000    700         150   150    70.00         15.00   15.00
Potato___healthy                                 152    106          23    23    69.74         15.13   15.13
Tomato___Bacterial_spot                         2127   1489         319   319    70.00         15.00   15.00
Tomato___Early_blight                           1000    700         150   150    70.00         15.00   15.00
Tomato___Late_blight                            1901   1331         285   285    70.02         14.99   14.99
Tomato___Leaf_Mold                               952    666         143   14

In [22]:
# --------------------------------------------------
# DATA LEAKAGE CHECK
# --------------------------------------------------

train_paths = set(train_df["path"])
val_paths = set(val_df["path"])
test_paths = set(test_df["path"])

# Check pairwise overlap
train_val_overlap = train_paths & val_paths
train_test_overlap = train_paths & test_paths
val_test_overlap = val_paths & test_paths

print("=" * 60)
print("DATA LEAKAGE CHECK")
print("=" * 60)

print("Train images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Test images:", len(test_paths))

print("\nOverlaps:")
print("Train ↔ Validation:", len(train_val_overlap))
print("Train ↔ Test:", len(train_test_overlap))
print("Validation ↔ Test:", len(val_test_overlap))

# Check total unique paths
all_paths = train_paths | val_paths | test_paths

print("\nTotal unique image paths:", len(all_paths))
print("Expected unique images:", len(df))

# Final result
leakage_free = (
    len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
    and len(all_paths) == len(df)
)

print("\n" + "=" * 60)
print(
    "DATA LEAKAGE STATUS:",
    "PASSED" if leakage_free else "FAILED"
)
print("=" * 60)

DATA LEAKAGE CHECK
Train images: 14208
Validation images: 3045
Test images: 3045

Overlaps:
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0

Total unique image paths: 20298
Expected unique images: 20298

DATA LEAKAGE STATUS: PASSED


In [23]:
# --------------------------------------------------
# Create physical train / val / test directories
# --------------------------------------------------

import shutil
from pathlib import Path

# Output directories
train_dir = SPLIT_DIR / "train"
val_dir = SPLIT_DIR / "val"
test_dir = SPLIT_DIR / "test"

# Create directories
for split_dir in [train_dir, val_dir, test_dir]:
    split_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Function to copy images
# --------------------------------------------------

def copy_split_images(split_df, destination_dir, split_name):

    copied = 0

    for _, row in split_df.iterrows():

        source_path = Path(row["path"])
        class_name = row["class"]

        class_destination = destination_dir / class_name
        class_destination.mkdir(parents=True, exist_ok=True)

        destination_path = class_destination / source_path.name

        shutil.copy2(source_path, destination_path)

        copied += 1

    print(f"{split_name}: {copied} images copied")
    
    return copied


# --------------------------------------------------
# Copy each split
# --------------------------------------------------

train_copied = copy_split_images(
    train_df,
    train_dir,
    "Train"
)

val_copied = copy_split_images(
    val_df,
    val_dir,
    "Validation"
)

test_copied = copy_split_images(
    test_df,
    test_dir,
    "Test"
)

# --------------------------------------------------
# Final copy count
# --------------------------------------------------

print("\n" + "=" * 60)
print("COPY SUMMARY")
print("=" * 60)

print("Train copied:", train_copied)
print("Validation copied:", val_copied)
print("Test copied:", test_copied)

print(
    "\nTotal copied:",
    train_copied + val_copied + test_copied
)

Train: 14208 images copied
Validation: 3045 images copied
Test: 3045 images copied

COPY SUMMARY
Train copied: 14208
Validation copied: 3045
Test copied: 3045

Total copied: 20298


In [24]:
# --------------------------------------------------
# PHYSICAL DATASET SPLIT VALIDATION
# --------------------------------------------------

def count_images_in_split(split_dir):
    
    counts = {}

    for class_dir in sorted(split_dir.iterdir()):
        
        if not class_dir.is_dir():
            continue

        count = sum(
            1
            for file in class_dir.iterdir()
            if file.is_file() and file.suffix.lower() in image_extensions
        )

        counts[class_dir.name] = count

    return counts


train_counts = count_images_in_split(train_dir)
val_counts = count_images_in_split(val_dir)
test_counts = count_images_in_split(test_dir)


# --------------------------------------------------
# Create validation table
# --------------------------------------------------

physical_distribution = pd.DataFrame({
    "Expected Train": train_df["class"].value_counts(),
    "Actual Train": pd.Series(train_counts),

    "Expected Validation": val_df["class"].value_counts(),
    "Actual Validation": pd.Series(val_counts),

    "Expected Test": test_df["class"].value_counts(),
    "Actual Test": pd.Series(test_counts)
}).fillna(0).astype(int)

print("=" * 80)
print("PHYSICAL DATASET VALIDATION")
print("=" * 80)

print(physical_distribution.sort_index().to_string())


# --------------------------------------------------
# Check whether expected == actual
# --------------------------------------------------

train_match = (
    physical_distribution["Expected Train"]
    == physical_distribution["Actual Train"]
).all()

val_match = (
    physical_distribution["Expected Validation"]
    == physical_distribution["Actual Validation"]
).all()

test_match = (
    physical_distribution["Expected Test"]
    == physical_distribution["Actual Test"]
).all()


# --------------------------------------------------
# Check total counts
# --------------------------------------------------

actual_train_total = sum(train_counts.values())
actual_val_total = sum(val_counts.values())
actual_test_total = sum(test_counts.values())

print("\n" + "=" * 80)
print("TOTAL COUNT VALIDATION")
print("=" * 80)

print("Expected Train:", len(train_df))
print("Actual Train:", actual_train_total)

print("\nExpected Validation:", len(val_df))
print("Actual Validation:", actual_val_total)

print("\nExpected Test:", len(test_df))
print("Actual Test:", actual_test_total)


# --------------------------------------------------
# Final status
# --------------------------------------------------

physical_split_valid = (
    train_match
    and val_match
    and test_match
    and actual_train_total == len(train_df)
    and actual_val_total == len(val_df)
    and actual_test_total == len(test_df)
)

print("\n" + "=" * 80)
print("FINAL PHYSICAL SPLIT STATUS")
print("=" * 80)

print(
    "Train distribution:",
    "PASS" if train_match else "FAIL"
)

print(
    "Validation distribution:",
    "PASS" if val_match else "FAIL"
)

print(
    "Test distribution:",
    "PASS" if test_match else "FAIL"
)

print(
    "\nOverall physical split:",
    "PASSED" if physical_split_valid else "FAILED"
)

PHYSICAL DATASET VALIDATION
                                               Expected Train  Actual Train  Expected Validation  Actual Validation  Expected Test  Actual Test
Potato___Early_blight                                     700           700                  150                150            150          150
Potato___Late_blight                                      700           700                  150                150            150          150
Potato___healthy                                          106           106                   23                 23             23           23
Tomato___Bacterial_spot                                  1489          1489                  319                319            319          319
Tomato___Early_blight                                     700           700                  150                150            150          150
Tomato___Late_blight                                     1331          1331                  285            

In [ ]:
# --------------------------------------------------
# STEP 5.1 — DATASET STRUCTURE VALIDATION
# --------------------------------------------------

from pathlib import Path

SPLIT_DIR = Path("../data/processed/tomato_potato_split")

split_paths = {
    "Train": SPLIT_DIR / "train",
    "Validation": SPLIT_DIR / "val",
    "Test": SPLIT_DIR / "test"
}

print("=" * 70)
print("TRAIN / VALIDATION / TEST STRUCTURE")
print("=" * 70)

expected_classes = set([
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___healthy",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus"
])

image_extensions = {".jpg", ".jpeg", ".png"}

expected_totals = {
    "Train": 14208,
    "Validation": 3045,
    "Test": 3045
}

all_structure_valid = True

for split_name, split_path in split_paths.items():

    print(f"\n{split_name}")
    print("-" * 50)

    exists = split_path.exists()
    print("Directory exists:", exists)

    if not exists:
        all_structure_valid = False
        continue

    actual_classes = {
        folder.name
        for folder in split_path.iterdir()
        if folder.is_dir()
    }

    missing_classes = expected_classes - actual_classes
    extra_classes = actual_classes - expected_classes

    print("Number of classes:", len(actual_classes))
    print("Missing classes:", missing_classes)
    print("Unexpected classes:", extra_classes)

    total_images = 0

    for class_name in sorted(expected_classes):

        class_path = split_path / class_name

        if not class_path.exists():
            all_structure_valid = False
            continue

        count = sum(
            1
            for file in class_path.iterdir()
            if file.is_file()
            and file.suffix.lower() in image_extensions
        )

        total_images += count

    print("Total images:", total_images)
    print("Expected images:", expected_totals[split_name])

    count_valid = total_images == expected_totals[split_name]
    class_valid = (
        len(actual_classes) == 13
        and len(missing_classes) == 0
        and len(extra_classes) == 0
    )

    print(
        "Class structure:",
        "PASS" if class_valid else "FAIL"
    )

    print(
        "Image count:",
        "PASS" if count_valid else "FAIL"
    )

    if not class_valid or not count_valid:
        all_structure_valid = False


print("\n" + "=" * 70)
print("STEP 5.1 FINAL STATUS")
print("=" * 70)

print(
    "Dataset structure:",
    "PASSED" if all_structure_valid else "FAILED"
)

TRAIN / VALIDATION / TEST STRUCTURE

Train
--------------------------------------------------
Directory exists: True
Number of classes: 13
Missing classes: set()
Unexpected classes: set()


In [ ]:
# --------------------------------------------------
# STEP 5.2 — BASIC IMAGE PREPROCESSING PIPELINE
# --------------------------------------------------

import torch
from torchvision import transforms
from PIL import Image
from pathlib import Path

# --------------------------------------------------
# Configuration
# --------------------------------------------------

IMAGE_SIZE = 224

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# --------------------------------------------------
# Validation / Test preprocessing
# No augmentation
# --------------------------------------------------

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# --------------------------------------------------
# Locate one training image
# --------------------------------------------------

TRAIN_DIR = Path("../data/processed/tomato_potato_split/train")

sample_images = list(TRAIN_DIR.rglob("*.jpg"))

print("Training directory exists:", TRAIN_DIR.exists())
print("JPEG images found:", len(sample_images))

# --------------------------------------------------
# Load one image
# --------------------------------------------------

sample_path = sample_images[0]

image = Image.open(sample_path).convert("RGB")

print("\nOriginal image:")
print("Path:", sample_path)
print("Size:", image.size)
print("Mode:", image.mode)

# --------------------------------------------------
# Apply preprocessing
# --------------------------------------------------

tensor = val_test_transform(image)

print("\nProcessed tensor:")
print("Shape:", tensor.shape)
print("Data type:", tensor.dtype)
print("Minimum:", tensor.min().item())
print("Maximum:", tensor.max().item())

# --------------------------------------------------
# Validation
# --------------------------------------------------

assert image.size == (256, 256)
assert tensor.shape == (3, 224, 224)
assert tensor.dtype == torch.float32

print("\n" + "=" * 60)
print("STEP 5.2 STATUS: PASSED")
print("=" * 60)

In [ ]:
# --------------------------------------------------
# STEP 5.3 — FINAL TRAINING AUGMENTATION
# --------------------------------------------------

from torchvision import transforms

IMAGE_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.85, 1.0),
        ratio=(0.9, 1.1)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print("Final training augmentation pipeline created.")

print("\nTraining pipeline:")
print("1. RandomResizedCrop: 85–100%")
print("2. RandomHorizontalFlip: p=0.5")
print("3. ColorJitter: mild")
print("4. ToTensor")
print("5. ImageNet normalization")

print("\nValidation/Test pipeline:")
print("1. Resize: 224×224")
print("2. ToTensor")
print("3. ImageNet normalization")

print("\n" + "=" * 60)
print("STEP 5.3 STATUS: PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 5.4 — VISUAL AUGMENTATION VERIFICATION
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import torch

# ------------------------------------------------------------
# Dataset path
# ------------------------------------------------------------

TRAIN_DIR = Path("data/processed/tomato_potato_split/train")

print("Training directory:")
print(TRAIN_DIR.resolve())
print("Exists:", TRAIN_DIR.exists())

# ------------------------------------------------------------
# Find image files
# ------------------------------------------------------------

image_extensions = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

sample_images = [
    file
    for file in TRAIN_DIR.rglob("*")
    if file.is_file() and file.suffix in image_extensions
]

print("Number of training images found:", len(sample_images))

# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

if len(sample_images) == 0:
    print("\nERROR: No training images were found.")
    print("Contents of training directory:")

    if TRAIN_DIR.exists():
        for item in list(TRAIN_DIR.iterdir())[:10]:
            print(" -", item)
    else:
        print("Training directory does not exist.")

    raise FileNotFoundError(
        "No training images found. Check the dataset path."
    )

# ------------------------------------------------------------
# Select one original image
# ------------------------------------------------------------

sample_path = sample_images[0]

print("\nSelected image:")
print(sample_path)

original_image = Image.open(sample_path).convert("RGB")

print("Original image size:", original_image.size)

# ------------------------------------------------------------
# Display original + augmented versions
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Original
axes[0, 0].imshow(original_image)
axes[0, 0].set_title("Original")
axes[0, 0].axis("off")

# ------------------------------------------------------------
# Generate 5 augmented versions
# ------------------------------------------------------------

for i in range(1, 6):

    augmented_tensor = train_transform(original_image)

    # Undo ImageNet normalization for visualization
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    display_tensor = torch.clamp(
        augmented_tensor * std + mean,
        0,
        1
    )

    display_image = (
        display_tensor
        .permute(1, 2, 0)
        .numpy()
    )

    row = i // 3
    col = i % 3

    axes[row, col].imshow(display_image)
    axes[row, col].set_title(f"Augmented {i}")
    axes[row, col].axis("off")

plt.suptitle(
    "Step 5.4 — Training Augmentation Verification",
    fontsize=14
)

plt.tight_layout()
plt.show()

print("\nSTEP 5.4 PASSED — Augmentation pipeline verified.")